# Project 1 · Seq2Seq Num2Text

## 데이터 처리

`D2-1_Seq2Seq.pdf` 15–18 `W07_word2vec-implementation.pdf` 8、11–14

In [1]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
DATA_FILE = PROJECT_DIR / "num2word.zip"
RETRAIN = False  # 저장 모델 사용

In [2]:
from collections import Counter, defaultdict
from io import TextIOWrapper
from random import Random
from zipfile import ZipFile


PAIRS = (("num", "en"), ("num", "ko"), ("en", "ko"))
EDGE_CASES = {
    "짧은 숫자": ["7"],
    "십의 배수": ["90"],
    "백의 배수": ["100", "900"],
    "연속된 0": ["1005", "3000"],
    "자릿수 경계": ["9999", "10000", "10001", "100000", "1000000", "10000000", "100000000", "1000000000"],
    "0이 포함된 긴 숫자": ["100000001", "1000000001"],
}
# 원본에 없는 예의 고정 정답은 추가 테스트에만 사용한다.
MANUAL_REFERENCES = {
    "10000": ("ten thousand", "만"),
    "100000": ("one hundred thousand", "십만"),
    "1000000": ("one million", "백만"),
    "10000000": ("ten million", "천만"),
    "100000000": ("one hundred million", "일억"),
    "1000000000": ("one billion", "십억"),
    "100000001": ("one hundred million one", "일억일"),
    "1000000001": ("one billion one", "십억일"),
}


def normalize(text, language):
    """Normalize whitespace without changing the meaning of a string."""
    if not isinstance(text, str):
        raise TypeError("Input must be a string.")
    if language == "num":
        text = text.strip()
        if not (1 <= len(text) <= 10 and text.isascii() and text.isdigit()):
            raise ValueError("Use 1–10 ASCII digits.")
        if len(text) > 1 and text.startswith("0"):
            raise ValueError("Leading zeroes are outside the corpus format.")
    elif language == "en":
        text = " ".join(text.lower().split())
    elif language == "ko":
        text = "".join(text.split())
    else:
        raise ValueError(f"Unknown language: {language}")
    if not text:
        raise ValueError("Input must not be empty.")
    return text


def read_corpus(path):
    """Join the files using the original digit string as the key."""
    corpus = {}
    with ZipFile(path) as archive:
        for language, filename in (("en", "num2txt.txt"), ("ko", "num2kor.txt")):
            with TextIOWrapper(archive.open(filename), encoding="utf-8-sig") as file:
                for line in file:
                    number, text = line.split("\t", maxsplit=1)
                    number = normalize(number, "num")
                    text = normalize(text, language)
                    # 같은 숫자는 한 레코드로 합친다.
                    row = corpus.setdefault(number, {"num": number})
                    if row.setdefault(language, text) != text:
                        raise ValueError(f"Conflicting {language} label for {number}")
    return corpus


def pair_coverage(corpus):
    """Count raw pairs by digit length before splitting."""
    return {pair: Counter(len(number) for number, row in corpus.items()
                          if all(language in row for language in pair))
            for pair in PAIRS}


def split_corpus(corpus, seed):
    """An approximately 80/10/10 partition shared by all six directions."""
    edge_numbers = {number for numbers in EDGE_CASES.values() for number in numbers}
    strata = defaultdict(list)
    for number, row in corpus.items():
        if number not in edge_numbers and number != "0":
            # 자릿수와 언어 쌍 보유 여부로 층화한다.
            strata[len(number), "en" in row, "ko" in row].append(number)

    rng = Random(seed)
    parts = {"train": [], "validation": [], "test": []}
    if "0" in corpus:
        # zero와 영은 한 번만 나오므로 학습에 둔다.
        parts["train"].append("0")
    for stratum in sorted(strata):
        numbers = sorted(strata[stratum])
        rng.shuffle(numbers)
        holdout = max(1, len(numbers) // 10) if len(numbers) >= 3 else 0
        parts["validation"].extend(numbers[:holdout])
        parts["test"].extend(numbers[holdout:2 * holdout])
        parts["train"].extend(numbers[2 * holdout:])
    return parts


def edge_references(corpus):
    """Yield the reserved raw labels and the separate manual probes."""
    for label, numbers in EDGE_CASES.items():
        for number in numbers:
            if number in MANUAL_REFERENCES:
                en, ko = MANUAL_REFERENCES[number]
                yield label, number, {"num": number, "en": en, "ko": ko}, "manual"
            else:
                yield label, number, corpus[number], "corpus"

In [3]:
SEED = 2026

corpus = read_corpus(DATA_FILE)
splits = split_corpus(corpus, SEED)
train_ids, validation_ids, test_ids = map(set, (splits["train"], splits["validation"], splits["test"]))
edge_ids = {number for numbers in EDGE_CASES.values() for number in numbers}
assert train_ids.isdisjoint(validation_ids | test_ids | edge_ids)
assert validation_ids.isdisjoint(test_ids | edge_ids)
assert test_ids.isdisjoint(edge_ids)
assert train_ids | validation_ids | test_ids | (edge_ids & corpus.keys()) == corpus.keys()

print("전체 숫자 개수")
print(" | ".join(f"{name} {len(ids):,}" for name, ids in splits.items()))
print(f"추가 원본 숫자 {len(edge_ids & corpus.keys())}")
print("\n원본 쌍 개수")
print("digits  num↔en  num↔ko   en↔ko")
coverage = pair_coverage(corpus)
for digits in range(1, 11):
    values = "  ".join(f"{coverage[pair][digits]:>6,}" for pair in PAIRS)
    print(f"{digits:>6}  {values}")

전체 숫자 개수
train 149,814 | validation 18,720 | test 18,720
추가 원본 숫자 8

원본 쌍 개수
digits  num↔en  num↔ko   en↔ko
     1      10      10      10
     2      90      90      90
     3     900     900     900
     4   9,000   9,000   9,000
     5  15,000  15,000   2,449
     6  15,000  15,000     256
     7  15,000  15,000      30
     8  15,000  15,000       3
     9  15,000  15,000       0
    10  15,000  15,000       0


## 모델

`D2-1_Seq2Seq.pdf` 24、27、56–69 `D1-2_RNNs (slides).pdf` 31–39 `D1-3_LSTM.pdf` 10–12

In [4]:
import math

import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, TensorDataset


PAD, BOS, EOS, UNK = range(4)
SPECIAL = ("<pad>", "<bos>", "<eos>", "<unk>", "<to:en>", "<to:ko>", "<to:num>")
MAX_STEPS = 32


def tokens(text, language):
    # 영어는 단어, 한글과 숫자는 문자
    parts = text.split() if language == "en" else text
    return [f"{language}:{part}" for part in parts]


class Vocabulary:
    def __init__(self, token_list):
        self.tokens = list(token_list)
        self.ids = {token: index for index, token in enumerate(self.tokens)}

    @classmethod
    def from_training(cls, corpus, train_ids, pair):
        observed = set()
        for number in train_ids:
            row = corpus[number]
            if all(language in row for language in pair):
                for language in pair:
                    observed.update(tokens(row[language], language))
        return cls([*SPECIAL, *sorted(observed)])

    def encode(self, text, language):
        return [self.ids.get(token, UNK) for token in tokens(text, language)]

    def source(self, text, language, target_language):
        # 목표 언어를 첫 토큰으로 표시
        return [self.ids[f"<to:{target_language}>"], *self.encode(text, language), EOS]

    def target(self, text, language):
        return [BOS, *self.encode(text, language), EOS]

    def allowed(self, language):
        return [EOS, *[i for i, token in enumerate(self.tokens) if token.startswith(f"{language}:")]]

    def decode(self, ids, language):
        end = ids.index(EOS) if EOS in ids else len(ids)
        parts = [self.tokens[index].split(":", 1)[1] for index in ids[:end]]
        return (" " if language == "en" else "").join(parts)


def pair_dataset(corpus, ids, pair, vocab):
    rows = [corpus[number] for number in ids if all(lang in corpus[number] for lang in pair)]
    datasets = []
    for source_lang, target_lang in (pair, pair[::-1]):
        source_ids = [torch.tensor(vocab.source(row[source_lang], source_lang, target_lang)) for row in rows]
        target_ids = [torch.tensor(vocab.target(row[target_lang], target_lang)) for row in rows]
        datasets.append(TensorDataset(pad_sequence(source_ids, batch_first=True),
                                      pad_sequence(target_ids, batch_first=True)))
    return datasets


class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size, padding_idx=PAD)
        self.gru = nn.GRU(embedding_size, hidden_size, batch_first=True)

    def forward(self, source):
        states, _ = self.gru(self.embedding(source))
        valid = source.ne(PAD)
        # 단방향 GRU의 마지막 유효 상태
        last = valid.sum(dim=1) - 1
        initial = states[torch.arange(source.size(0), device=source.device), last]
        return states, valid, initial.unsqueeze(0)


class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size, padding_idx=PAD)
        self.gru = nn.GRU(embedding_size, hidden_size, batch_first=True)
        self.output = nn.Sequential(nn.Linear(2 * hidden_size, hidden_size), nn.Tanh(),
                                    nn.Linear(hidden_size, vocab_size))

    def forward(self, previous, hidden, encoder_states, valid):
        query, hidden = self.gru(self.embedding(previous), hidden)
        # Luong 내적 점수에서 PAD 제외
        scores = torch.bmm(query, encoder_states.transpose(1, 2)) / math.sqrt(query.size(-1))
        weights = scores.masked_fill(~valid[:, None, :], -torch.inf).softmax(dim=-1)
        context = torch.bmm(weights, encoder_states)
        return self.output(torch.cat((query, context), dim=-1)), hidden


class Seq2Seq(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size):
        super().__init__()
        self.encoder = Encoder(vocab_size, embedding_size, hidden_size)
        self.decoder = Decoder(vocab_size, embedding_size, hidden_size)

    def forward(self, source, decoder_input):
        states, valid, hidden = self.encoder(source)
        logits, _ = self.decoder(decoder_input, hidden, states, valid)
        return logits

    @torch.inference_mode()
    def greedy(self, source, allowed_ids, max_steps=MAX_STEPS):
        states, valid, hidden = self.encoder(source)
        allowed = torch.tensor(allowed_ids, device=source.device)
        previous = source.new_full((source.size(0), 1), BOS)
        finished = torch.zeros(source.size(0), dtype=torch.bool, device=source.device)
        generated = []
        for _ in range(max_steps):
            logits, hidden = self.decoder(previous, hidden, states, valid)
            # 최고 점수 토큰을 다시 입력
            predicted = allowed[logits[:, 0, allowed].argmax(dim=-1)]
            predicted = predicted.masked_fill(finished, EOS)
            generated.append(predicted)
            finished |= predicted.eq(EOS)
            if finished.all():
                break
            previous = predicted[:, None]
        return torch.stack(generated, dim=1)

## 학습

`D2-1_Seq2Seq.pdf` 29–31

In [5]:
import time


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(2)

BATCH_SIZE = 512
EMBED_SIZE = 32
HIDDEN_SIZE = 96
EPOCHS = {("num", "en"): 16, ("num", "ko"): 16, ("en", "ko"): 44}
CHECKPOINT_VERSION = 2


def model_path(directory, pair):
    return Path(directory) / ("_".join(pair) + ".pt")


@torch.inference_mode()
def exact_matches(model, dataset, vocab, target_lang):
    model.eval()
    allowed = vocab.allowed(target_lang)
    correct = []
    for source, target in DataLoader(dataset, batch_size=BATCH_SIZE):
        predicted = model.greedy(source.to(DEVICE), allowed)
        expected = target[:, 1:].to(DEVICE)
        predicted = nn.functional.pad(predicted, (0, expected.size(1) - predicted.size(1)))
        # EOS까지 모두 같아야 정답
        correct.extend((predicted.eq(expected) | expected.eq(PAD)).all(dim=1).tolist())
    return correct


def fit_pair(corpus, splits, pair, directory, epochs=None):
    torch.manual_seed(SEED + PAIRS.index(pair))
    vocab = Vocabulary.from_training(corpus, splits["train"], pair)
    train = pair_dataset(corpus, splits["train"], pair, vocab)
    validation = pair_dataset(corpus, splits["validation"], pair, vocab)
    loaders = [DataLoader(data, batch_size=BATCH_SIZE, shuffle=True) for data in train]
    model = Seq2Seq(len(vocab.tokens), EMBED_SIZE, HIDDEN_SIZE).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    epochs = EPOCHS[pair] if epochs is None else epochs
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, [max(1, epochs * 3 // 4)], gamma=1 / 3)
    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)
    best = -1.0
    path = model_path(directory, pair)
    for epoch in range(1, epochs + 1):
        started = time.perf_counter()
        model.train()
        loss_sum = 0.0
        token_count = 0
        # 두 방향을 같은 빈도로 학습
        for paired_batches in zip(*loaders):
            for source, target in paired_batches:
                source, target = source.to(DEVICE), target.to(DEVICE)
                optimizer.zero_grad()
                # 정답의 다음 토큰 예측
                decoder_input, expected = target[:, :-1], target[:, 1:]
                logits = model(source, decoder_input)
                loss = loss_fn(logits.flatten(0, 1), expected.flatten())
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                count = expected.ne(PAD).sum().item()
                loss_sum += loss.item() * count
                token_count += count

        scores = []
        for data, language in zip(validation, (pair[1], pair[0])):
            matches = exact_matches(model, data, vocab, language)
            scores.append(sum(matches) / len(matches))
        score = sum(scores) / len(scores)
        scheduler.step()
        if score > best:
            # 검증 정확도로만 저장
            best = score
            checkpoint = {"version": CHECKPOINT_VERSION, "pair": list(pair), "seed": SEED,
                          "epoch": epoch, "validation": scores, "vocab": vocab.tokens,
                          "embedding_size": EMBED_SIZE, "hidden_size": HIDDEN_SIZE,
                          "state_dict": model.state_dict()}
            temporary_path = path.with_suffix(".tmp")
            torch.save(checkpoint, temporary_path)
            temporary_path.replace(path)
        print(f"{' ↔ '.join(pair)} | {epoch:02d}/{epochs} | CE {loss_sum / token_count:.4f} | "
              f"val {scores[0]:.2%}, {scores[1]:.2%} | {time.perf_counter() - started:.1f}s", flush=True)


def load_pair(directory, pair):
    checkpoint = torch.load(model_path(directory, pair), map_location=DEVICE, weights_only=True)
    if (checkpoint["version"], checkpoint["pair"], checkpoint["seed"]) != (
            CHECKPOINT_VERSION, list(pair), SEED):
        raise ValueError(f"Checkpoint version, direction or split seed mismatch for {pair}.")
    vocab = Vocabulary(checkpoint["vocab"])
    model = Seq2Seq(len(vocab.tokens), checkpoint["embedding_size"], checkpoint["hidden_size"]).to(DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    return model.eval(), vocab, checkpoint

In [6]:
if RETRAIN:
    for pair in PAIRS:
        fit_pair(corpus, splits, pair, PROJECT_DIR)
else:
    print("저장된 모델을 마지막 셀에서 불러옵니다.")

저장된 모델을 마지막 셀에서 불러옵니다.


## 디코딩과 평가

`D2-1_Seq2Seq.pdf` 40–41

In [7]:
def predict(model, vocab, text, source_lang, target_lang):
    text = normalize(text, source_lang)
    source = torch.tensor([vocab.source(text, source_lang, target_lang)], device=DEVICE)
    ids = model.greedy(source, vocab.allowed(target_lang))[0].tolist()
    return vocab.decode(ids, target_lang), EOS in ids


def translate(text, source_lang, target_lang):
    """Convert a text string between 'num', 'en', and 'ko'."""
    pair = next((pair for pair in PAIRS if (source_lang, target_lang) in (pair, pair[::-1])), None)
    if pair is None:
        raise ValueError("Choose two different language codes from 'num', 'en', 'ko'.")
    model, vocab, _ = models[pair]
    prediction, finished = predict(model, vocab, text, source_lang, target_lang)
    if not finished:
        raise RuntimeError("Greedy decoding did not reach EOS within the length limit.")
    return prediction


def report_test(corpus, test_ids, models):
    print("원본 테스트 세트 · Exact match (incl. EOS)")
    accuracies = []
    for pair, (model, vocab, _) in models.items():
        pair_ids = [number for number in test_ids if all(lang in corpus[number] for lang in pair)]
        datasets = pair_dataset(corpus, pair_ids, pair, vocab)
        for dataset, (src, tgt) in zip(datasets, (pair, pair[::-1])):
            matches = exact_matches(model, dataset, vocab, tgt)
            correct, total = sum(matches), len(matches)
            accuracies.append(correct / total)
            print(f"{src} → {tgt}  {correct}/{total}  {correct / total:.2%}")
            if pair == ("en", "ko"):
                buckets = {size: [ok for number, ok in zip(pair_ids, matches) if len(number) == size]
                           for size in (8, 9, 10)}
                parts = [f"{size} {sum(group)}/{len(group)}" if group else f"{size} N/A (0 samples)"
                         for size, group in buckets.items()]
                print("    " + " · ".join(parts))
    print(f"6방향 평균  {sum(accuracies) / len(accuracies):.2%}")


def report_edges(corpus, models):
    directions = [direction for pair in PAIRS for direction in (pair, pair[::-1])]
    print("\n추가 테스트")
    print("            " + "  ".join(f"{src}→{tgt}" for src, tgt in directions))
    summaries = {"corpus": [0, 0], "manual": [0, 0]}
    examples = []
    for label, number, references, status in edge_references(corpus):
        marks = []
        for src, tgt in directions:
            if src not in references or tgt not in references:
                marks.append("–")
                continue
            pair = (src, tgt) if (src, tgt) in models else (tgt, src)
            model, vocab, _ = models[pair]
            result, finished = predict(model, vocab, references[src], src, tgt)
            correct = finished and result == references[tgt]
            marks.append("✓" if correct else "×")
            summaries[status][0] += int(correct)
            summaries[status][1] += 1
            if not correct and (number in ("10000", "100000000", "1000000000")):
                eos_note = "" if finished else " [no EOS]"
                examples.append(f"  {number} {src}→{tgt}  {result!r}{eos_note}  |  {references[tgt]!r}")
        print(f"{number:10}  {status:6}  " + "      ".join(marks) + f"   {label}")
    for status, (correct, total) in summaries.items():
        print(f"  {status}: {correct}/{total}")
    if examples:
        print("경계 오답 예시")
        print("\n".join(examples))

## 사용 방법

`RETRAIN = False` `translate(text, source_lang, target_lang)`

In [8]:
# 1. 저장한 모델 불러오기
models = {pair: load_pair(PROJECT_DIR, pair) for pair in PAIRS}
for pair, (_, _, checkpoint) in models.items():
    scores = ", ".join(f"{score:.2%}" for score in checkpoint["validation"])
    print(f"{' ↔ '.join(pair)}  |  epoch {checkpoint['epoch']}  |  validation {scores}")

# 2. 테스트 및 경계 사례
print()
report_test(corpus, splits["test"], models)
report_edges(corpus, models)

# 3. 사용 예
print("\n사용 예 · translate(text, source_lang, target_lang)")
for text, source, target in (
    ("123", "num", "en"), ("one hundred twenty three", "en", "num"),
    ("123", "num", "ko"), ("백이십삼", "ko", "num"),
    ("one hundred twenty three", "en", "ko"), ("백이십삼", "ko", "en"),
):
    print(f"{source} → {target}  {text!r} → {translate(text, source, target)!r}")

num ↔ en  |  epoch 13  |  validation 99.86%, 99.91%
num ↔ ko  |  epoch 14  |  validation 99.92%, 99.96%
en ↔ ko  |  epoch 32  |  validation 99.37%, 99.76%

원본 테스트 세트 · Exact match (incl. EOS)


num → en  9986/9995  99.91%


en → num  9988/9995  99.93%


num → ko  9993/9995  99.98%


ko → num  9993/9995  99.98%
en → ko  1257/1270  98.98%
    8 0/1 · 9 N/A (0 samples) · 10 N/A (0 samples)
ko → en  1263/1270  99.45%
    8 0/1 · 9 N/A (0 samples) · 10 N/A (0 samples)
6방향 평균  99.70%

추가 테스트
            num→en  en→num  num→ko  ko→num  en→ko  ko→en
7           corpus  ✓      ✓      ✓      ✓      ✓      ✓   짧은 숫자
90          corpus  ✓      ✓      ✓      ✓      ✓      ✓   십의 배수
100         corpus  ✓      ✓      ✓      ✓      ✓      ✓   백의 배수
900         corpus  ✓      ✓      ✓      ✓      ✓      ✓   백의 배수
1005        corpus  ✓      ✓      ✓      ✓      ✓      ✓   연속된 0
3000        corpus  ✓      ✓      ✓      ✓      ✓      ✓   연속된 0
9999        corpus  ✓      ✓      ✓      ✓      ✓      ✓   자릿수 경계
10000       manual  ×      ✓      ×      ✓      ✓      ×   자릿수 경계
10001       corpus  ✓      ✓      ✓      ✓      ✓      ×   자릿수 경계
100000      manual  ×      ✓      ×      ✓      ×      ×   자릿수 경계
1000000     manual  ×      ×      ×      ✓      ×      ×   자릿수 경계
10000000    manu